# Dynamic Society Friction Simulator — A100 Training
**Target:** A100 40GB | **Budget:** ~4 hours (~30 compute units)

Run: **Runtime → Run all**


## Step 1 — Mount Drive & Check GPU


In [ ]:
!pip install -q pyyaml psutil
import os, time, json, shutil
from pathlib import Path
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_AVAILABLE = True
    print('Drive mounted')
except Exception:
    DRIVE_AVAILABLE = False
    print('Drive not available')

if not torch.cuda.is_available():
    print('WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> A100')

GPU_NAME    = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU:  {GPU_NAME}')
print(f'VRAM: {GPU_VRAM_GB:.1f} GB')
print(f'PyTorch: {torch.__version__}')

class Budget:
    RATE = {'A100':7.35,'H100':15.0,'L4':2.35,'T4':1.67}
    def __init__(self, total=100):
        self.total = total
        self.t0    = time.time()
        gpu        = next((k for k in self.RATE if k in GPU_NAME), 'A100')
        self.rate  = self.RATE[gpu]
        print(f'Budget: {total} units @ {self.rate}/hr = {total/self.rate:.1f}h total')
    def used(self):   return (time.time()-self.t0)/3600 * self.rate
    def left(self):   return max(0, self.total - self.used())
    def status(self): print(f'Budget used: {self.used():.2f}/{self.total} units | {self.left():.1f} remaining')

budget = Budget(total=100)


## Step 2 — Clone Repo & Install


In [ ]:
import os, subprocess

REPO_URL    = 'https://github.com/vivek797029/Dynamic-Societal-Friction-Simulator.git'
PROJECT_DIR = '/content/Dynamic-Societal-Friction-Simulator'

if not os.path.exists(PROJECT_DIR):
    print('Cloning...')
    subprocess.run(['git','clone',REPO_URL,PROJECT_DIR], check=True)
else:
    print('Pulling latest...')
    subprocess.run(['git','-C',PROJECT_DIR,'pull'], check=True)

os.chdir(PROJECT_DIR)
print(f'CWD: {os.getcwd()}')
print('Installing deps...')
!pip install -e '.[dev,eval]' -q
print('Installing Flash Attention 2...')
!pip install flash-attn --no-build-isolation -q
print('All installed')


## Step 3 — Verify Config


In [ ]:
import os, yaml
os.chdir('/content/Dynamic-Societal-Friction-Simulator')

with open('configs/model_config.yaml') as f:
    cfg = yaml.safe_load(f)

tcfg = cfg['training']
eff  = tcfg['per_device_train_batch_size'] * tcfg['gradient_accumulation_steps']
print('Config loaded:')
print(f"  Model:   {cfg['base_model']['name']}")
print(f"  Epochs:  {tcfg['num_train_epochs']}")
print(f"  Batch:   {tcfg['per_device_train_batch_size']} x {tcfg['gradient_accumulation_steps']} = {eff}")
print(f"  SeqLen:  {tcfg['max_seq_length']}")
print(f"  LoRA r:  {cfg['lora']['r']}")
print(f"  bf16:    {tcfg['bf16']}")

cfg['gdrive']['enabled'] = DRIVE_AVAILABLE
with open('configs/model_config.yaml','w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
print('Config ready')


## Step 4 — Generate Training Data


In [ ]:
import os
os.chdir('/content/Dynamic-Societal-Friction-Simulator')

if os.path.exists('data/processed/train.jsonl'):
    n = sum(1 for _ in open('data/processed/train.jsonl'))
    print(f'Data exists: {n:,} samples')
else:
    print('Generating 50k samples...')
    !dsfs generate-data --num-samples 50000 --output-dir data/processed --seed 42
    print('Done')


## Step 5 — W&B


In [ ]:
import os
os.environ['WANDB_DISABLED'] = 'true'
print('W&B disabled')


## Step 6 — TRAIN


In [ ]:
import os, logging
os.chdir('/content/Dynamic-Societal-Friction-Simulator')
logging.basicConfig(level=logging.INFO)

from src.model.trainer import train

budget.status()
try:
    train(config_path='configs/model_config.yaml')
    print('TRAINING COMPLETE')
except KeyboardInterrupt:
    print('Interrupted')
except Exception as e:
    print(f'Error: {e}')
    raise
finally:
    budget.status()


## Step 7 — Budget Status


In [ ]:
budget.status()


## Step 8 — Test Model


In [ ]:
import os
from pathlib import Path
os.chdir('/content/Dynamic-Societal-Friction-Simulator')

from src.model.inference import FrictionLLM

adapter = Path('./outputs/checkpoints/final_adapter')
if not adapter.exists():
    print(f'No adapter at {adapter.absolute()}')
else:
    llm    = FrictionLLM(config_path='configs/model_config.yaml', adapter_path=str(adapter))
    result = llm.analyze_friction('Tensions between residents and immigrants. Politician blames immigrants for housing costs.')
    print(result)


## Step 9 — Save to Drive


In [ ]:
import os, shutil
from pathlib import Path
os.chdir('/content/Dynamic-Societal-Friction-Simulator')

adapter = Path('./outputs/checkpoints/final_adapter')
if not adapter.exists():
    print('No adapter found')
else:
    mb = sum(f.stat().st_size for f in adapter.rglob('*') if f.is_file()) / (1024**2)
    print(f'Adapter: {mb:.1f} MB')
    if DRIVE_AVAILABLE:
        dst = Path('/content/drive/MyDrive/dsfs-final-adapter')
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(adapter, dst)
        print(f'Saved to Drive: {dst}')
    budget.status()
